# Setup Imports and Notebook Environment

In [ ]:
import math

import cartopy.crs as ccrs
import datashader as dsh
import geoviews as gv
import holoviews as hv
import hvplot.xarray  # noqa: F401
import ipywidgets as widgets
import numpy as np
import panel as pn
import param
import xarray as xr
import geoviews as gv
from geoviews import opts
from holoviews import opts
from holoviews.operation.datashader import rasterize, shade
from IPython.display import HTML, display
import warnings
import datacube

In [ ]:
gv.extension("bokeh")

opts.defaults(
    opts.Image(width=600, height=400, colorbar=True),
    opts.RGB(width=600, height=400),
    opts.Feature(apply_ranges=False),
    opts.QuadMesh(width=600, height=400, colorbar=True),
)

# ODC 1.9 needs to clean up it's warnings. They're not useful for users here.
warnings.filterwarnings('ignore', category=datacube.migration.ODC2DeprecationWarning)

%matplotlib inline

In [ ]:
dc = datacube.Datacube()

# Search and Create Dask Array

In [ ]:
# Setup time and AOI
query = {
    "latitude": (-23.3249, -23.6382),
    "longitude": (151.7381, 152.0964),
    "time": "2024-04-11",  # The one day there's a clear image
}

ds = dc.load(
    product="enmap_hsi_l2a",
    output_crs="EPSG:32756",
    resolution=30,
    driver="rio",  # Critical to enable Hyperspectral data loading
    dask_chunks={},  # Load lazily (to check the data size first and to load in parallel)
    **query
)
display(ds)

# Load into RAM

In [ ]:
%%time
# Load into RAM and remove the time dimension
ds = ds.load().squeeze()

# Draw on a map as an RGB Image

This is really rough, it's just picking a single wavelength for Red, Green and Blue and applying a bit of scaling.

Lots of room for improvement, but does give some representation of the data.

In [ ]:
def normalize(band):
    band_min, band_max = (np.nanmin(band), np.nanmax(band))
    return ((band-band_min)/((band_max - band_min)))


def brighten(band):
    alpha = 1
    beta = 0
    return np.clip(alpha*band + beta, 0, 3_000)


def get_rgb(ds):
    if isinstance(ds, xr.Dataset):
        da = ds.squeeze()['image']
    else:
        da = ds

    # Wavelengths consistent with Sentinel-2 R, G, B
    r = da.sel(wavelength=655, method="nearest")
    g = da.sel(wavelength=560, method="nearest")
    b = da.sel(wavelength=492, method="nearest")

    r_b = brighten(r)
    g_b = brighten(g)
    b_b = brighten(b)

    r_bn = normalize(r_b)
    g_bn = normalize(g_b)
    b_bn = normalize(b_b)

    return gv.RGB((r.x, r.y, r_bn, g_bn, b_bn), crs=ccrs.epsg(32756))
    # return gv.RGB((r.x, r.y, r, g, b), crs=ccrs.epsg(32756))


tiles = hv.element.tiles.CartoLight()

true_colour = get_rgb(ds)
tiles * true_colour

# Hover over map to display the spectrum

Use some HoloViews to start visualising the captured spectrum. Move the cursor across the map, or select a map tool to zoom in/out.

In [ ]:
true_colour = get_rgb(ds)

posxy = hv.streams.PointerXY(source=true_colour, x=np.nan, y=np.nan)

# make a function that displays the location when called.
def location_str(x, y):
    """Display pane showing the x and y values"""
    return pn.pane.Str('Click at %0.3f, %0.3f' % (x, y), width=200)


def hover_spectrum(x, y):
    wavelengths = ds.image.sel(x=x, y=y, method='nearest')

    return hv.Curve(wavelengths)

hover_dmap = hv.DynamicMap(hover_spectrum, streams=[posxy])

# Set specific options to ensure the x-axis scales properly
hover_dmap = hover_dmap.opts(
    opts.Curve(
        width=700, 
        height=600,
        # framewise=True,     # Recompute limits are computed for each frame
        ylim=(0,3000),
        xlim=(400,2500),
    )
)

pn.Row(
    true_colour.opts(width=600, height=600),
    hover_dmap
)

# Click to Select Point for Spectrum Comparison

More complex HoloViews visualisation to allow comparing two points. Click to select a point, Click again to move the point, hover to compare.

In [ ]:
true_colour = get_rgb(ds)

# Create a global store to persist the selected point across callbacks
class PointStore(param.Parameterized):
    x = param.Number(default=np.nan)
    y = param.Number(default=np.nan)
    has_selection = param.Boolean(default=False)

# Initialize our store
point_store = PointStore()

# Stream for tracking mouse position
hover_stream = hv.streams.PointerXY(source=true_colour, x=np.nan, y=np.nan)

# Stream for tracking clicked points
tap_stream = hv.streams.Tap(source=true_colour, x=np.nan, y=np.nan)


# Function to get spectrum for current cursor position
def get_hover_spectrum(x, y):
    """Generate the spectrum curve for the current hover position"""
    if np.isnan(x) or np.isnan(y):
        return hv.Curve([], kdims=['Wavelength'], vdims=['Value']).opts(
            line_color="#0000ff", line_width=2)
    
    try:
        wavelengths = ds.image.sel(x=x, y=y, method='nearest')
        return hv.Curve(wavelengths, label='Hover Position').opts(
            line_color="#0000ff", line_width=2)
    except Exception as e:
        print(f"Error getting hover spectrum: {e}")
        return hv.Curve([], kdims=['Wavelength'], vdims=['Value']).opts(
            line_color="#0000ff", line_width=2)

# Function to get spectrum for the stored (clicked) position
def get_stored_spectrum():
    """Generate the spectrum curve for the stored position"""
    if not point_store.has_selection or np.isnan(point_store.x) or np.isnan(point_store.y):
        return hv.Curve([], kdims=['Wavelength'], vdims=['Value'])
    
    try:
        wavelengths = ds.image.sel(x=point_store.x, y=point_store.y, method='nearest')
        return hv.Curve(wavelengths, label='Selected Point').opts(
            line_color="#ff0000", line_width=2)
    except Exception as e:
        print(f"Error getting stored spectrum: {e}")
        return hv.Curve([], kdims=['Wavelength'], vdims=['Value'])

# Function to create the combined visualization
def combined_visualization(x, y):
    """Create overlay of hover and stored spectra"""
    hover_curve = get_hover_spectrum(x, y)
    
    if point_store.has_selection:
        stored_curve = get_stored_spectrum()
        # Create an overlay with both curves
        return hv.NdOverlay({
            'Hover Position': hover_curve,
            'Selected Point': stored_curve
        }, kdims=['Curve']).opts(show_legend=True)
    else:
        # Just return the hover curve if no point is selected
        return hv.NdOverlay({
            'Hover Position': hover_curve,
        
        }, kdims=['Curve']).opts(show_legend=True)
        

# Create a Points element to show the selected point
def selected_point_marker(x, y):
    """Create a Points element for the selected location"""
    # if not np.isnan(x) and not np.isnan(y):
    #     point_store.update(x, y)
    #     return gv.Points([(x, y)], crs=ccrs.epsg(32756)).opts(color='red', size=10, marker='x')
    # else:
    #     return gv.Points([])
    if not np.isnan(x) and not np.isnan(y):
        point_store.x = x
        point_store.y = y
        point_store.has_selection = True
    
    if not point_store.has_selection:
        return gv.Points([])

    point = gv.Points([(point_store.x, point_store.y)], crs=ccrs.epsg(32756)).opts(color="#ff6666", size=16, marker='x')
    return point


# Create dynamic maps
spectrum_dmap = hv.DynamicMap(combined_visualization, streams=[hover_stream])
point_dmap = hv.DynamicMap(selected_point_marker, streams=[tap_stream])

# Set options for the spectrum plot
spectrum_opts = {
    'width': 700,
    'height': 600,
    'framewise': True,
    'show_grid': True,
    'xlim': (400, 2500),
    'ylim': (0, 3000),
}

spectrum_dmap = spectrum_dmap.opts(**spectrum_opts)

# Combine the true color image with the point overlay
map_view = (true_colour * point_dmap).opts(width=600, height=600)

# Create a Panel text pane for coordinate display
coord_pane = pn.pane.Str("Hover over the image to see coordinates")

# Update the coordinate pane when mouse moves
def update_coords(x, y):
    if np.isnan(x) or np.isnan(y):
        coord_pane.object = "Hover over the image to see coordinates"
    else:
        coord_pane.object = f"Mouse at {x:.3f}, {y:.3f}"


hover_stream.add_subscriber(update_coords)

# Create the layout
layout = pn.Row(
    pn.Column(map_view, coord_pane),
    spectrum_dmap,
    # point_dmap
)

layout

# Apply (very) a basic mask to remove dubious data

In [ ]:
masked_image = ds.image.where((ds.quality_testflags == 0) & (ds.defective_pixel_mask == 0))

true_colour = get_rgb(masked_image)
true_colour

# Review Available Masks (Defective Pixels, Cloud, Haze, etc)

For (much) more detail see the Product Specification linked from the top of this notebook.

The ODC masking functions aren't currently working with Hyperspectral data loading, but the quality layers supplied are reasonably easy to work with.

## Quality Test Flags

Non-zero for failing one of the quality checks.


In [ ]:
(ds.quality_testflags != 0).plot.imshow(size=6, levels=2)

## Defective Pixel Mask

This catches quite a lot of low quality pixels!

In [ ]:
ds.defective_pixel_mask.isel(x=500, y=500)

In [ ]:
total_pixels = ds.defective_pixel_mask.count().item()
defective_pixels = ds.defective_pixel_mask.sum().item()
percent_defective = defective_pixels / total_pixels

print(f'Out of {total_pixels:,} pixels, {defective_pixels:,} or {percent_defective:.2%} are defective')

### Plot defective pixels by Space and by Wavelength

In [ ]:
ds.defective_pixel_mask.sum(dim='wavelength').plot.imshow(size=8, cmap='viridis')

In [ ]:
ds.defective_pixel_mask.sum(dim=('x', 'y')).plot(size=8)

# Cloud Mask

No cloud or cloud shadow detected in this image.

In [ ]:
ds.quality_cloud.sum().item()

In [ ]:
ds.quality_cloud_shadow.sum().item()

## Quality Layer - Haze

In [ ]:
ds.quality_haze.sum().item()

In [ ]:
ds.quality_haze.plot.imshow(size=6, levels=2)

# Quality Classes

Quality classes are:

|Value|Class|
|----|-----|
|0|None|
|1|Land|
|2|Water|
|3|Background|

In [ ]:
ds.quality_classes.plot.imshow(size=6, levels=4)